# Lasso Regression Example

This notebook demonstrates Lasso regression using a small synthetic housing dataset.
It implements coordinate descent from scratch, so the example runs without extra packages.

Lasso adds an L1 penalty, which can shrink less useful coefficients all the way to zero.

In [ ]:
import math
import random

In [ ]:
random.seed(7)

feature_names = ["size_sqft", "rooms", "distance_to_city", "age"]

X = []
y = []

for _ in range(40):
    size_sqft = random.uniform(600, 2500)
    rooms = random.uniform(1, 6)
    distance_to_city = random.uniform(1, 30)
    age = random.uniform(1, 40)
    noise = random.uniform(-15000, 15000)

    price = (
        180 * size_sqft
        + 22000 * rooms
        - 3500 * distance_to_city
        + noise
    )

    X.append([size_sqft, rooms, distance_to_city, age])
    y.append(price)

print("First 3 samples:")
for i in range(3):
    row = {name: round(value, 2) for name, value in zip(feature_names, X[i])}
    print(f"X[{i}] =", row, "-> y =", round(y[i], 2))

In [ ]:
def mean(values):
    return sum(values) / len(values)


def column(matrix, j):
    return [row[j] for row in matrix]


def mse(actual, predicted):
    return sum((a - p) ** 2 for a, p in zip(actual, predicted)) / len(actual)

In [ ]:
def standardize_features(matrix):
    means = []
    stds = []
    scaled = [[0.0] * len(matrix[0]) for _ in range(len(matrix))]

    for j in range(len(matrix[0])):
        values = column(matrix, j)
        mu = mean(values)
        variance = sum((value - mu) ** 2 for value in values) / len(values)
        sigma = math.sqrt(variance) or 1.0

        means.append(mu)
        stds.append(sigma)

        for i in range(len(matrix)):
            scaled[i][j] = (matrix[i][j] - mu) / sigma

    return scaled, means, stds


def center_target(values):
    mu = mean(values)
    return [value - mu for value in values], mu


X_scaled, x_means, x_stds = standardize_features(X)
y_centered, y_mean = center_target(y)

In [ ]:
def soft_threshold(value, penalty):
    if value > penalty:
        return value - penalty
    if value < -penalty:
        return value + penalty
    return 0.0


def lasso_coordinate_descent(matrix, target, alpha=1.0, epochs=200):
    n_samples = len(matrix)
    n_features = len(matrix[0])
    weights = [0.0] * n_features

    for _ in range(epochs):
        for j in range(n_features):
            residual = []

            for i in range(n_samples):
                prediction_without_j = sum(
                    matrix[i][k] * weights[k]
                    for k in range(n_features)
                    if k != j
                )
                residual.append(target[i] - prediction_without_j)

            rho = sum(matrix[i][j] * residual[i] for i in range(n_samples))
            z = sum(matrix[i][j] ** 2 for i in range(n_samples))

            weights[j] = soft_threshold(rho, alpha) / z if z else 0.0

    return weights


def recover_coefficients(scaled_weights, means, stds, y_mean):
    coefficients = [scaled_weights[j] / stds[j] for j in range(len(scaled_weights))]
    intercept = y_mean - sum(coefficients[j] * means[j] for j in range(len(coefficients)))
    return intercept, coefficients


def predict(matrix, intercept, coefficients):
    return [
        intercept + sum(row[j] * coefficients[j] for j in range(len(coefficients)))
        for row in matrix
    ]

In [ ]:
alpha = 25000
scaled_weights = lasso_coordinate_descent(X_scaled, y_centered, alpha=alpha, epochs=250)
intercept, coefficients = recover_coefficients(scaled_weights, x_means, x_stds, y_mean)
predictions = predict(X, intercept, coefficients)

print("Chosen alpha:", alpha)
print("Intercept:", round(intercept, 2))
print("\nCoefficients:")
for name, value in zip(feature_names, coefficients):
    print(f"{name}: {value:.4f}")

print("\nTraining MSE:", round(mse(y, predictions), 2))

In [ ]:
print("Coefficient shrinkage as alpha increases:\n")

for alpha in [1000, 10000, 25000, 40000]:
    scaled_weights = lasso_coordinate_descent(X_scaled, y_centered, alpha=alpha, epochs=250)
    _, coeffs = recover_coefficients(scaled_weights, x_means, x_stds, y_mean)
    rounded = {name: round(value, 4) for name, value in zip(feature_names, coeffs)}
    print(f"alpha = {alpha}: {rounded}")

In [ ]:
new_house = [1800, 4, 8, 20]
predicted_price = intercept + sum(value * coeff for value, coeff in zip(new_house, coefficients))

print("Prediction for a new house:")
print({name: value for name, value in zip(feature_names, new_house)})
print("Predicted price:", round(predicted_price, 2))

With a larger `alpha`, Lasso shrinks weak features more aggressively.
In this example, the `age` coefficient eventually becomes exactly `0.0`, showing how L1 regularization can perform feature selection.